In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import pyvista as pv
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot


In [ ]:
import pickle

with open(Path.cwd() / 'noise_samples.pkl', 'rb') as f:
    noise_samples = pickle.load(f)

stage_names = list(noise_samples.keys())
print(f"{len(stage_names)} stages: {stage_names}")
for name in stage_names:
    arr = np.asarray(noise_samples[name])
    print(f"{name:25s} n={arr.size:>8d}  mean={arr.mean():>12.3f}  std={arr.std():>12.3f}")


In [ ]:
from scipy import stats

n_stages = len(stage_names)
ncols = 3
nrows = int(np.ceil(n_stages / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, name in zip(axes, stage_names):
    arr = np.asarray(noise_samples[name]).ravel()
    mu, sigma = arr.mean(), arr.std()

    ax.hist(arr, bins=200, density=True, color='steelblue', alpha=0.7, label='samples')

    x = np.linspace(arr.min(), arr.max(), 400)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=1.5, label='Gaussian fit')

    ax.set_title(name, fontsize=10)
    ax.set_xlabel('amplitude')
    ax.set_ylabel('density')

for ax in axes[n_stages:]:
    ax.axis('off')

axes[0].legend(fontsize=8)
fig.suptitle('Noise amplitude distribution by processing stage', y=1.02, fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
final_stage = stage_names[-1]
arr = np.asarray(noise_samples[final_stage]).ravel()

loc_laplace, scale_laplace = stats.laplace.fit(arr)
mu_gauss, sigma_gauss = arr.mean(), arr.std()

print(f"Stage: {final_stage}")
print(f"Laplace fit:  loc={loc_laplace:.3f}  scale={scale_laplace:.3f}")
print(f"Gaussian fit: mu={mu_gauss:.3f}  sigma={sigma_gauss:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(arr, bins=200, density=True, color='steelblue', alpha=0.7, label='samples')

x = np.linspace(arr.min(), arr.max(), 400)
ax.plot(x, stats.laplace.pdf(x, loc_laplace, scale_laplace), 'g-', lw=1.5, label='Laplace fit')
ax.plot(x, stats.norm.pdf(x, mu_gauss, sigma_gauss), 'r--', lw=1.5, label='Gaussian fit')

ax.set_title(f'Noise distribution at "{final_stage}" with Laplace fit')
ax.set_xlabel('amplitude')
ax.set_ylabel('density')
ax.legend()
fig.tight_layout()
plt.show()


# Noise from last stage

In [ ]:
noise_model = {
    'distribution': 'laplace',
    'stage': final_stage,
    'loc': loc_laplace,
    'scale': scale_laplace,
    'n_samples_fit': arr.size,
}

model_path = Path.cwd() / 'laplace_noise_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(noise_model, f)

print(f"Saved noise model to {model_path}")
print(noise_model)


# Noise before gain

In [ ]:
pre_gain_stage = '7 Constant Velocity'   # last stage before "8 Spherical Gain"
arr_pre_gain = np.asarray(noise_samples[pre_gain_stage]).ravel()

loc_laplace_pre, scale_laplace_pre = stats.laplace.fit(arr_pre_gain)
mu_gauss_pre, sigma_gauss_pre = arr_pre_gain.mean(), arr_pre_gain.std()

print(f"Stage: {pre_gain_stage}")
print(f"Laplace fit:  loc={loc_laplace_pre:.3f}  scale={scale_laplace_pre:.3f}")
print(f"Gaussian fit: mu={mu_gauss_pre:.3f}  sigma={sigma_gauss_pre:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(arr_pre_gain, bins=200, density=True, color='steelblue', alpha=0.7, label='samples')

x = np.linspace(arr_pre_gain.min(), arr_pre_gain.max(), 400)
ax.plot(x, stats.laplace.pdf(x, loc_laplace_pre, scale_laplace_pre), 'g-', lw=1.5, label='Laplace fit')
ax.plot(x, stats.norm.pdf(x, mu_gauss_pre, sigma_gauss_pre), 'r--', lw=1.5, label='Gaussian fit')

ax.set_title(f'Noise distribution at "{pre_gain_stage}" (pre-gain) with Laplace fit')
ax.set_xlabel('amplitude')
ax.set_ylabel('density')
ax.legend()
fig.tight_layout()
plt.show()

# ── Save pre-gain noise model ───────────────────────────────────────────────
noise_model_pre_gain = {
    'distribution': 'laplace',
    'stage': pre_gain_stage,
    'loc': loc_laplace_pre,
    'scale': scale_laplace_pre,
    'n_samples_fit': arr_pre_gain.size,
}

model_path_pre_gain = Path.cwd() / 'laplace_noise_model_pre_gain.pkl'
with open(model_path_pre_gain, 'wb') as f:
    pickle.dump(noise_model_pre_gain, f)

print(f"Saved noise model to {model_path_pre_gain}")
print(noise_model_pre_gain)


# Migrating Pure Noise

Sanity check for the rest of this study: migrate an **empty B-scan plus noise only** —
no scatterers, no real signal at all — through Kirchhoff, Gazdag, and back-propagation
(the same `helper_functions/migration.py` functions used throughout
`TimeLapse_Playground.ipynb`/`TimeLapse_Processing.ipynb`). If any method turns pure
noise into something that *looks* like a coherent, scatterer-like focus, that's a
false-positive risk for every noisy result elsewhere in this project — this establishes
what the "noise floor" actually looks like after each migration method.

No tapering/t0-shift preprocessing is applied here (unlike the real B-scans elsewhere):
there's no wavelet or hyperbola tail to suppress in pure noise, so the raw noise samples
go straight into migration.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))
from helper_functions.migration import PylopsKirchoffMigration, gazdag_migration, write_backprop_files

# ── Reuse the real study's grid + physics constants (no new geometry to invent) ──
TIMELAPSE_ROOT = _here / 'timelapse_study'

d_static = np.load(str(TIMELAPSE_ROOT / 'static_results.npz'), allow_pickle=False)
time_ns        = d_static['time_ns']
dt             = float(d_static['dt'])
x_traces       = d_static['x_traces']
ref_signal_std = float(d_static['data_static'][0].std())   # Baseline scenario, for noise scaling

d_mig = np.load(str(TIMELAPSE_ROOT / 'migrated_results.npz'), allow_pickle=False)
z_img = d_mig['z_img']

n_t, n_traces = len(time_ns), len(x_traces)
dt_ns = dt * 1e9

v_ice = 0.168    # m/ns
f_c   = 1.5      # GHz
lam   = v_ice / f_c
kz_c  = 2 * np.pi / lam
t0_ns = np.sqrt(2) / f_c   # Ricker peak delay -- kept for snapshot-timing consistency only, see markdown above

dz_mig = float(z_img[1] - z_img[0])
dx_mig = float(x_traces[1] - x_traces[0])

print(f"Grid:  n_t={n_t}, n_traces={n_traces}, dt_ns={dt_ns:.6f}, dz_mig={dz_mig*1e3:.2f} mm, dx_mig={dx_mig*1e3:.2f} mm")
print(f"Reference (Baseline) signal std = {ref_signal_std:.4f}  (used only to scale the injected noise)")

In [ ]:
NOISE_LEVEL = 0.1   # same convention as TimeLapse_Playground's "Create Noisy Data" section
rng = np.random.default_rng(0)

empty_bscan = np.zeros((n_t, n_traces))

target_std   = NOISE_LEVEL * ref_signal_std
scaled_scale = target_std / np.sqrt(2)   # Var(Laplace) = 2 * scale**2
pure_noise_bscan = empty_bscan + stats.laplace.rvs(
    loc=noise_model_pre_gain['loc'], scale=scaled_scale,
    size=empty_bscan.shape, random_state=rng,
)

print(f"Pure-noise B-scan: shape={pure_noise_bscan.shape}, std={pure_noise_bscan.std():.4f} "
      f"(target {target_std:.4f}, {NOISE_LEVEL:.0%} of the reference signal std)")

fig, ax = plt.subplots(figsize=(8, 5))
vmax = np.abs(pure_noise_bscan).max()
im = ax.imshow(pure_noise_bscan, aspect='auto',
               extent=[x_traces[0], x_traces[-1], time_ns[-1], 0],
               cmap='seismic', vmin=-vmax, vmax=vmax, interpolation='nearest')
ax.set_xlabel('x [m]'); ax.set_ylabel('Time [ns]')
ax.set_title('Pure-Noise "B-scan"  (empty, scatterer-free B-scan + Laplace noise only)')
fig.colorbar(im, ax=ax, label='Ez [a.u.]')
plt.tight_layout()
plt.show()

## Kirchhoff & Gazdag Migration of Pure Noise

Both are pure post-processing on the noise array above — fast, no gprMax run needed.

In [ ]:
ANGLE_AP = 40   # same Kirchhoff aperture as TimeLapse_Playground

# Kirchhoff wants (n_tr, n_t); Gazdag wants (n_t, n_x) -- pure_noise_bscan is
# already (n_t, n_traces), so only Kirchhoff needs the transpose.
bscan_tr_first = pure_noise_bscan.T   # (n_traces, n_t)

print('Running Kirchhoff migration on pure noise...', end=' ', flush=True)
t0 = _time.perf_counter()
mig_kirchhoff_noise = PylopsKirchoffMigration(
    bscan_tr_first, time_ns, x_traces, v_ice, z_img, f0=f_c, angleaperture=ANGLE_AP
)
print(f'{_time.perf_counter()-t0:.1f} s  shape={mig_kirchhoff_noise.shape}')

print('Running Gazdag migration on pure noise...', flush=True)
t0 = _time.perf_counter()
mig_gazdag_noise = gazdag_migration(pure_noise_bscan, x_traces, time_ns, z_img, v_ice)
print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={mig_gazdag_noise.shape}')

In [ ]:
extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]
extent_mig   = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

vmax_b = np.abs(pure_noise_bscan).max()
axes[0].imshow(pure_noise_bscan, aspect='auto', extent=extent_bscan, cmap='seismic',
               vmin=-vmax_b, vmax=vmax_b, interpolation='nearest')
axes[0].set_title('Pure-noise B-scan (input)')
axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('Time [ns]')
fig.colorbar(im, ax=axes[0], label='Ez [V/m]')

vmax_k = np.abs(mig_kirchhoff_noise).max()
axes[1].imshow(mig_kirchhoff_noise, aspect='auto', extent=extent_mig, cmap='seismic',
               vmin=-vmax_k, vmax=vmax_k, origin='upper')
axes[1].set_title('Kirchhoff migration of pure noise')
axes[1].set_xlabel('x [m]'); axes[1].set_ylabel('Depth z [m]')
fig.colorbar(im, ax=axes[1], label='Ez [V/m]')

vmax_g = np.abs(mig_gazdag_noise).max()
axes[2].imshow(mig_gazdag_noise, aspect='auto', extent=extent_mig, cmap='seismic',
               vmin=-vmax_g, vmax=vmax_g, origin='upper')
axes[2].set_title('Gazdag migration of pure noise')
axes[2].set_xlabel('x [m]'); axes[2].set_ylabel('Depth z [m]')
fig.colorbar(im, ax=axes[2], label='Ez [V/m]')

fig.suptitle('Migrating Pure Noise (no scatterers, no signal) — Kirchhoff vs Gazdag',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Result:** the two methods do not fail the same way. **Kirchhoff turns pure noise into
clearly coherent, smooth wave-like bands** that could easily be misread as real layered
structure — its delay-and-sum aperture stacking imposes coherence on incoherent input by
construction (summing many traces along travel-time curves smooths incoherent noise into
locally-correlated structure, especially with no taper here to suppress the far-aperture
contributions that drive this). **Gazdag's noise output stays speckled and incoherent**
— no wave-like artefacts, just texture. This is a real difference in false-positive risk
between the two methods, not just a cosmetic one: a real noisy B-scan's Kirchhoff
migration could plausibly contain Kirchhoff-induced coherent structure indistinguishable
from genuine reflectors, whereas Gazdag's noise floor stays visually "noise-like".

## Back-Propagation of Pure Noise

Unlike Kirchhoff/Gazdag, back-propagation isn't a direct post-processing step — the
noise *is* the excitation: `write_backprop_files` time-reverses `bscan_tr_first` and
writes it as a gprMax source file, so seeing it migrated means actually running gprMax
on the generated `.in` file (same split as every other back-propagation section in this
project: no cell here calls `api()` directly — run the `.in` file externally, then come
back and re-run the load/plot cell below).

Two variants are generated side by side, to test directly on the purest possible case
(zero signal, 100% noise) the same question raised in `TimeLapse_Processing.ipynb`:
does **sign-bit** excitation (`sign_bit=True` — inject `sign(u)`, stripping amplitude
entirely) suppress spurious focusing from noise spikes acting as their own competing
point sources, compared to the default peak-normalised excitation?

One simplification: `t0_ns` (the Ricker wavelet's peak delay, used only to time the
focus-window snapshots) doesn't strictly apply to pure noise — there's no wavelet here.
It's kept anyway so the snapshot timing lines up with the rest of this project's
back-propagation runs for a fair comparison.

In [ ]:
NOISE_STUDY_ROOT = _here / 'noise_study'
NOISE_STUDY_ROOT.mkdir(parents=True, exist_ok=True)

eps_r = 3.15   # same ice permittivity as the rest of this project
STRIDE, N_SNAP, SNAP_WIN = 1, 30, 1.0

bp_paths_noise = {}
for label, slug, sign_bit in [
    ('Pure Noise (peak-normalised)', 'purenoise_peaknorm', False),
    ('Pure Noise (sign-bit)',        'purenoise_signbit',  True),
]:
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        NOISE_STUDY_ROOT, label, slug, bscan_tr_first, dt_ns, x_traces,
        t0_ns, eps_r, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN,
        sign_bit=sign_bit
    )
    bp_paths_noise[slug] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'[{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB')
    print(f'  {in_path}')

print('\nNOTE: .in files only -- run each through gprMax externally to produce the '
      '.vti snapshots the cell below expects, e.g.:\n'
      f'  python -m gprMax {bp_paths_noise["purenoise_peaknorm"].relative_to(_here)}\n'
      f'  python -m gprMax {bp_paths_noise["purenoise_signbit"].relative_to(_here)}')

In [ ]:
def load_backprop_focus_frame(slug, n_t, dt_ns, t0_ns, n_snap, snap_win):
    """Mirrors TimeLapse_Playground.ipynb's focus-frame loader -- find the snapshot
    closest to t_focus = T - t0 and return its Ez component."""
    T_ns       = n_t * dt_ns
    t_focus_ns = T_ns - t0_ns
    t_start_ns = max(0.0, t_focus_ns - snap_win)
    dt_s       = dt_ns * 1e-9
    snap_step  = max(1, int((T_ns * 1e-9 - t_start_ns * 1e-9) / (max(1, n_snap - 1) * dt_s)))

    snap_dir   = NOISE_STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))
    if not snap_files:
        print(f'[{slug}] No snapshots in {snap_dir.name} -- run the .in file through gprMax first.')
        return None

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4, len(snap_files) - 1)

    snaps_ez = []
    for p in snap_files:
        mesh   = pv.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))   # domain/cell size fixed by write_backprop_files

    return {'ez': np.stack(snaps_ez)[idx_focus], 't_actual': snap_times_ns[idx_focus]}


focus_peaknorm = load_backprop_focus_frame('purenoise_peaknorm', n_t, dt_ns, t0_ns, N_SNAP, SNAP_WIN)
focus_signbit  = load_backprop_focus_frame('purenoise_signbit',  n_t, dt_ns, t0_ns, N_SNAP, SNAP_WIN)

if focus_peaknorm is None and focus_signbit is None:
    print('\nNo back-propagation snapshots available yet -- run the .in files above '
          'through gprMax, then re-run this cell.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
    for ax, frame, title in zip(axes, [focus_peaknorm, focus_signbit],
                                 ['Peak-normalised excitation', 'Sign-bit excitation']):
        if frame is None:
            ax.text(0.5, 0.5, 'not yet run', transform=ax.transAxes, ha='center', va='center')
            ax.set_title(title)
            continue
        vmax = np.percentile(np.abs(frame['ez']), 99.5)
        ax.imshow(frame['ez'], aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
                  extent=[0, 4.0, 0, 1], origin='lower')
        ax.set_title(f"{title}\nt={frame['t_actual']:.2f} ns")
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        fig.colorbar(im, ax=ax, label='Ez [V/m]')
    fig.suptitle('Back-Propagation of Pure Noise — focus-time snapshot',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()